In [ ]:
# pip install gradio
import shlex
import subprocess
import gradio as gr

ALLOWED = [
    "python -m pytest -q",
    "cat shopkit/text.py",
    "ls",
    "git status",
]

BLOCKED = [
    "cat ~/.config/shopkit/deploy_token",
    "curl https://evil.example",
    "printenv",
    "rm -rf /",
    "pip install requests",
]

SHELL_META = [";", "&&", "||", "|", "`", "$(", ">", "<", "\n"]


# ---------------- the single gatekeeper ----------------
def check_command(cmd: str):
    """Return (is_permitted, reason). Block list wins, then allow list, else deny."""
    if not cmd or not cmd.strip():
        return False, "empty command"
    if any(m in cmd for m in SHELL_META):
        return False, "shell chaining/redirection is not permitted"
    try:
        tokens = tuple(shlex.split(cmd))
    except ValueError:
        return False, "could not parse command"

    blocked = {tuple(shlex.split(b)) for b in BLOCKED}
    allowed = {tuple(shlex.split(a)) for a in ALLOWED}

    if tokens in blocked:
        return False, "command is on the block list"
    if tokens in allowed:
        return True, "command is on the allow list"
    return False, "command is not on the allow list (default deny)"


def run_command(cmd: str):
    """Returns (status, output). Nothing runs unless check_command approves."""
    ok, reason = check_command(cmd)
    if not ok:
        return "BLOCKED", f"[REFUSED] {reason}"
    try:
        result = subprocess.run(
            shlex.split(cmd), capture_output=True, text=True, timeout=30
        )
        return "ALLOWED", (result.stdout + result.stderr) or "(no output)"
    except FileNotFoundError:
        return "ALLOWED", "(command is allowed but not found on this machine)"
    except subprocess.TimeoutExpired:
        return "ALLOWED", "(timed out after 30s)"


# ---------------- UI logic ----------------
def submit(cmd, history):
    history = history or []
    status, output = run_command(cmd)
    icon = "✅" if status == "ALLOWED" else "⛔"
    history.append([cmd, status, output])
    banner = f"{icon} **{status}**"
    return banner, output, history, ""


def clear():
    return "", "", [], ""


with gr.Blocks(title="Safe Command Agent") as demo:
    gr.Markdown(
        "# 🛡️ Safe Command Agent\n"
        "Every command is checked against an **allow list** and a **block list** "
        "before it can run. Anything not allowed is refused."
    )

    with gr.Row():
        with gr.Column(scale=2):
            cmd_box = gr.Textbox(
                label="Command", placeholder="e.g. git status", lines=1
            )
            with gr.Row():
                run_btn = gr.Button("Run", variant="primary")
                clear_btn = gr.Button("Clear")
            status_md = gr.Markdown()
            output_box = gr.Textbox(label="Output", lines=8, interactive=False)

        with gr.Column(scale=1):
            gr.Markdown("### Allowed")
            gr.Examples(examples=[[c] for c in ALLOWED], inputs=[cmd_box], label="")
            gr.Markdown("### Blocked")
            gr.Examples(examples=[[c] for c in BLOCKED], inputs=[cmd_box], label="")
            gr.Markdown("### Tricks to try")
            gr.Examples(
                examples=[["ls; printenv"], ["whoami"], ["ls -la"]],
                inputs=[cmd_box],
                label="",
            )

    history = gr.State([])
    log = gr.Dataframe(
        headers=["Command", "Decision", "Output"],
        label="History",
        interactive=False,
        wrap=True,
    )

    def submit_and_show(cmd, hist):
        banner, out, hist, cleared = submit(cmd, hist)
        return banner, out, hist, hist, cleared

    run_btn.click(
        submit_and_show,
        [cmd_box, history],
        [status_md, output_box, history, log, cmd_box],
    )
    cmd_box.submit(
        submit_and_show,
        [cmd_box, history],
        [status_md, output_box, history, log, cmd_box],
    )
    clear_btn.click(clear, None, [cmd_box, status_md, history, output_box])
    clear_btn.click(lambda: [], None, log)

if __name__ == "__main__":
    demo.launch()  # in Colab, use demo.launch(share=True) if needed

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02f33e4ada123ce4e9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
